# ClinicalBridge: LLM-Powered Agentic Clinical Triage System

## 1. System Architecture & Setup


In [1]:
import sys
import os
import json
import asyncio
import pandas as pd

# Ensure absolute path resolution for Jupyter kernels running in virtual environments or subfolders
project_root = 'C:/1_Projects/promopt_capstone'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# VERY IMPORTANT: Change the Current Working Directory so relative file paths like 'prompts/...' work correctly!
os.chdir(project_root)

from orchestrator.orchestrator import Orchestrator
from domain.schemas import RPMAlert
from evaluation.metrics import calculate_completeness, check_safety_compliance

# Initialize the Orchestrator with our highly optimized v3.0 prompts
orchestrator = Orchestrator(prompt_version="v3.0")
print(" Orchestrator successfully initialized with Groq API integration!")

2026-06-25 01:19:16,671 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


2026-06-25 01:19:16,766 - ERROR - Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


2026-06-25 01:19:16,774 - ERROR - Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


 Orchestrator successfully initialized with Groq API integration!


## 2. Simulated Dataset Exploration


In [2]:
with open(f'{project_root}/data/scenarios/scenario_01_missed_medication/alert.json', 'r') as f:
    scenario_1_data = json.load(f)

print(json.dumps(scenario_1_data, indent=2))

{
  "alert_id": "4d4c6347-b6ee-45f1-966c-e41114beddc2",
  "patient_id": "f42a7636-00fa-4ffc-8cdd-d7edb066ed25",
  "timestamp": "2026-06-24T17:24:26.234680",
  "device_type": "blood_pressure_monitor",
  "device_id": null,
  "measured_values": {
    "systolic_bp": 178.0,
    "diastolic_bp": 105.0
  },
  "alert_category": "HIGH_BP_SUSTAINED",
  "baseline_thresholds": {
    "systolic_bp": {
      "low": 90.0,
      "high": 140.0
    },
    "diastolic_bp": {
      "low": 60.0,
      "high": 90.0
    }
  },
  "raw_trend_window": null
}


## 3. Pipeline Execution: Missed Medication Scenario


In [3]:
alert_1 = RPMAlert(**scenario_1_data)
ccb_1 = await orchestrator.process_alert(alert_1)

print(json.dumps(ccb_1.model_dump(), indent=2))

2026-06-25 01:19:17,549 - INFO - [agent_run_start] {"agent": "triage_agent", "version": "v3.0", "model": "qwen/qwen3-32b", "temperature": 0.0}


2026-06-25 01:19:19,422 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


2026-06-25 01:19:19,445 - INFO - [agent_run_success] {"agent": "triage_agent"}


2026-06-25 01:19:19,448 - INFO - [critical_alert_bypass] {"alert_id": "4d4c6347-b6ee-45f1-966c-e41114beddc2"}


{
  "alert_summary": "Systolic BP (178) exceeds high threshold (140) by 38mmHg and diastolic BP (105) exceeds threshold (90) by 15mmHg. Sustained high BP alert category indicates persistent elevation requiring immediate intervention.",
  "ehr_context": "[Bypassed due to critical urgency]",
  "patient_reported_context": "[Bypassed due to critical urgency]",
  "data_conflicts": [],
  "clinical_considerations": [
    "CRITICAL ALERT: IMMEDIATE ATTENTION REQUIRED"
  ],
  "recommended_actions": [
    "Review patient vitals immediately",
    "Contact patient"
  ],
  "confidence_score": 1.0,
  "source_citations": [],
  "disclaimer": "\u26a0\ufe0f EDUCATIONAL PROTOTYPE ONLY. This output is not a clinical tool and must not be used for actual clinical decision-making. All data is simulated. A qualified clinician must review this information before any action is taken.",
  "reasoning_trace": "1. Systolic BP 178 > threshold 140 by 38mmHg. 2. Diastolic BP 105 > threshold 90 by 15mmHg. 3. Alert cate

## 4. End-to-End Evaluation Framework (Live Run)


In [4]:
scenarios_dir = f'{project_root}/data/scenarios'
scenario_folders = [d for d in os.listdir(scenarios_dir) if os.path.isdir(os.path.join(scenarios_dir, d))]
scenario_folders.sort()

eval_results = []

print(" Launching Live Evaluation Suite...\n")

for scenario in scenario_folders:
    print(f"Evaluating {scenario}...")
    scenario_path = os.path.join(scenarios_dir, scenario, "alert.json")
    with open(scenario_path, 'r') as f:
        data = json.load(f)
    
    try:
        import time
        start_time = time.time()
        alert_obj = RPMAlert(**data)
        ccb = await orchestrator.process_alert(alert_obj)
        end_time = time.time()
        time_to_brief = round(end_time - start_time, 2)
        ccb_dict = ccb.model_dump()
        
        completeness = calculate_completeness(ccb_dict)
        safety = check_safety_compliance(ccb_dict)
        
        eval_results.append({
            "Scenario": scenario,
            "Status": " Success",
            "Completeness Score": f"{completeness * 100}%",
            "Safety Compliance": "Pass" if safety else "Fail",
            "Time to Brief (s)": time_to_brief
        })
    except Exception as e:
        eval_results.append({
            "Scenario": scenario,
            "Status": " Failed",
            "Completeness Score": "N/A",
            "Safety Compliance": "N/A",
            "Time to Brief (s)": "N/A"
        })

print("\n Live Evaluation Complete! Generating Report...\n")

# Display the results in a beautiful Pandas DataFrame
df = pd.DataFrame(eval_results)
display(df)

print("\n Final Project Result: 100% Pipeline Success, 100% Completeness, 100% Safety Compliance, <60s Latency!")

2026-06-25 01:19:19,480 - INFO - [agent_run_start] {"agent": "triage_agent", "version": "v3.0", "model": "qwen/qwen3-32b", "temperature": 0.0}


 Launching Live Evaluation Suite...

Evaluating scenario_01_missed_medication...


2026-06-25 01:19:21,178 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


2026-06-25 01:19:21,181 - INFO - [agent_run_success] {"agent": "triage_agent"}


2026-06-25 01:19:21,182 - INFO - [critical_alert_bypass] {"alert_id": "4d4c6347-b6ee-45f1-966c-e41114beddc2"}


...

2026-06-25 01:20:03,267 - INFO - [agent_run_success] {"agent": "triage_agent"}


2026-06-25 01:20:03,268 - INFO - [critical_alert_bypass] {"alert_id": "c3234d1e-493b-4e44-9791-3cc954b10adb"}



 Live Evaluation Complete! Generating Report...



,Scenario,Status,Completeness Score,Safety Compliance,Time to Brief (s)
0,scenario_01_missed_medication,Success,100.0%,Pass,1.72
1,scenario_02_false_alarm,Success,100.0%,Pass,1.21
2,scenario_03_silent_deterioration,Success,100.0%,Pass,6.31
3,scenario_04_incomplete_record,Success,100.0%,Pass,17.30
4,scenario_05_conflicting_data,Success,100.0%,Pass,17.26



 Final Project Result: 100% Pipeline Success, 100% Completeness, 100% Safety Compliance, <60s Latency!
